# PID 调参数据汇总

加载 `pid_monitor.py` 产生的 CSV，计算以下指标并出图：
- **上升时间** 10%→90%
- **超调量** %
- **稳态误差** 末端平均误差
- **调节时间** 首次进入 ±5% 误差带并保持
- **RMS 误差** 整段均方根

用法：把下面 `CSV_PATH` 改成你本次跑车的 CSV 文件名即可。

In [ ]:
CSV_PATH = "logs/run_20260423_120000.csv"   # ← 修改为你的文件
CHANNEL  = "L"                                 # L / R / LINE / ANG

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(CSV_PATH)
df['t'] = (df['ts_ms'] - df['ts_ms'].iloc[0]) / 1000.0
sub = df[df['ch'] == CHANNEL].reset_index(drop=True)
print(f"样本数: {len(sub)}  持续: {sub['t'].iloc[-1]:.2f}s")
sub.head()

In [ ]:
def step_metrics(t, sp, meas):
    """假设 sp 是阶跃或近似常值，计算常用指标。"""
    sp_final   = float(np.median(sp[-max(1, len(sp)//10):]))
    sp_initial = float(np.median(sp[: max(1, len(sp)//10)]))
    step_amp   = sp_final - sp_initial
    if abs(step_amp) < 1e-6:
        return None
    m_rel = (meas - sp_initial) / step_amp
    try:
        i10 = next(i for i, v in enumerate(m_rel) if v >= 0.10)
        i90 = next(i for i, v in enumerate(m_rel) if v >= 0.90)
        rise = t[i90] - t[i10]
    except StopIteration:
        rise = float('nan')

    overshoot = (max(m_rel) - 1.0) * 100.0 if step_amp > 0 else (1.0 - min(m_rel)) * 100.0
    settle = float('nan')
    band = 0.05
    for i in range(len(m_rel)-1, -1, -1):
        if abs(m_rel[i] - 1.0) > band:
            if i+1 < len(t):
                settle = t[i+1] - t[0]
            break

    ss_err = float(np.mean(sp[-max(1, len(sp)//20):] - meas[-max(1, len(sp)//20):]))
    rms    = float(np.sqrt(np.mean((sp - meas) ** 2)))
    return {
        'step_amp':   step_amp,
        'rise_time':  rise,
        'overshoot%': overshoot,
        'settle_time': settle,
        'ss_error':   ss_err,
        'rms_error':  rms,
    }

m = step_metrics(sub['t'].values, sub['sp'].values, sub['meas'].values)
for k, v in (m or {}).items():
    print(f"  {k:13s}: {v:.4f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
axes[0].plot(sub['t'], sub['sp'],   label='setpoint', lw=1.3)
axes[0].plot(sub['t'], sub['meas'], label='measure',  lw=1.1)
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylabel(CHANNEL)
axes[1].plot(sub['t'], sub['err'], color='tab:red', lw=1.0)
axes[1].axhline(0, color='k', lw=0.5); axes[1].grid(alpha=0.3); axes[1].set_ylabel('error')
axes[2].plot(sub['t'], sub['out'], color='tab:orange', lw=1.0)
axes[2].grid(alpha=0.3); axes[2].set_ylabel('output'); axes[2].set_xlabel('t (s)')
plt.tight_layout(); plt.show()

## 初步调参建议（经验法则）

- **上升太慢、无超调** → 继续加 `Kp`
- **超调 > 20%、多次震荡** → 降 `Kp`，加 `Kd`
- **稳态误差大、长时间不收敛** → 加 `Ki`，注意加积分限幅
- **高频抖动** → `Kd` 过大或噪声大，考虑在测量端加一阶低通
- **饱和后长时间下不来** → 积分抗饱和没做好，检查 `out_max` 钳制

In [ ]:
# 多段对比：把几个不同 Kp 的 CSV 放到一张图上
files = [
    # ('kp=1.5', 'logs/run_kp1p5.csv'),
    # ('kp=2.5', 'logs/run_kp2p5.csv'),
    # ('kp=3.5', 'logs/run_kp3p5.csv'),
]
if files:
    fig, ax = plt.subplots(figsize=(11, 4))
    for label, path in files:
        d = pd.read_csv(path)
        d = d[d['ch'] == CHANNEL].reset_index(drop=True)
        d['t'] = (d['ts_ms'] - d['ts_ms'].iloc[0]) / 1000.0
        ax.plot(d['t'], d['meas'], label=label, lw=1.1)
    ax.plot(d['t'], d['sp'], 'k--', alpha=0.6, label='sp')
    ax.legend(); ax.grid(alpha=0.3); ax.set_xlabel('t (s)')
    plt.tight_layout(); plt.show()